In [2]:
import pandas as pd
import numpy as np
import re

# =========================
# CONFIG (edit paths)
# =========================
TRAIN_IN  = "Preprocessed Datasets/cic-ids2017_common_features_train.csv"
TEST_IN   = "Preprocessed Datasets/cic-ids2017_common_features_test.csv"
TRAIN_OUT = "cic_ids2017_common_train_clean.csv"
TEST_OUT  = "cic_ids2017_common_test_clean.csv"

COMMON_FEATURES = [
    "flow_duration",
    "protocol",
    "service_state",
    "fwd_packet_count",
    "bwd_packet_count",
    "fwd_bytes",
    "bwd_bytes",
    "packet_rate",
    "byte_rate",
    "avg_packet_size",
    "syn_flag_count",
    "rst_flag_count",
    "ack_flag_count",
]

LABEL_COL = "label"  # if present

# =========================
# Cleaning helpers
# =========================
def normalize_str_series(s: pd.Series) -> pd.Series:
    s = s.astype("string")
    s = s.fillna("unknown")
    s = s.str.strip().str.lower()
    s = s.str.replace(r"\s+", "_", regex=True)
    s = s.str.replace(r"[^a-z0-9_\-]", "", regex=True)
    s = s.replace({"": "unknown", "nan": "unknown", "none": "unknown"})
    return s

def to_numeric_clean(df: pd.DataFrame, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def clip_nonnegative(df: pd.DataFrame, cols):
    for c in cols:
        if c in df.columns:
            df[c] = df[c].where(df[c] >= 0, 0)
    return df

def replace_inf_nan(df: pd.DataFrame):
    df = df.replace([np.inf, -np.inf], np.nan)
    return df

def cap_outliers_iqr(train: pd.DataFrame, test: pd.DataFrame, cols, k=3.0):
    """
    Cap outliers using TRAIN-only IQR bounds to avoid leakage.
    bounds: [Q1 - k*IQR, Q3 + k*IQR]
    """
    bounds = {}
    for c in cols:
        if c not in train.columns:
            continue
        x = train[c].dropna()
        if len(x) == 0:
            continue
        q1 = x.quantile(0.25)
        q3 = x.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0 or not np.isfinite(iqr):
            lo, hi = x.min(), x.max()
        else:
            lo = q1 - k * iqr
            hi = q3 + k * iqr
        bounds[c] = (float(lo), float(hi))

    for c, (lo, hi) in bounds.items():
        train[c] = train[c].clip(lower=lo, upper=hi)
        if c in test.columns:
            test[c] = test[c].clip(lower=lo, upper=hi)

    return train, test, bounds

def validate_schema(df: pd.DataFrame):
    missing = [c for c in COMMON_FEATURES if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

# =========================
# Load
# =========================
train = pd.read_csv(TRAIN_IN)
test  = pd.read_csv(TEST_IN)

validate_schema(train)
validate_schema(test)

# =========================
# Basic cleanup
# =========================
train = replace_inf_nan(train)
test  = replace_inf_nan(test)

# Normalize categorical fields
for c in ["protocol", "service_state"]:
    train[c] = normalize_str_series(train[c])
    test[c]  = normalize_str_series(test[c])

# Clean numeric fields
numeric_cols = [c for c in COMMON_FEATURES if c not in ("protocol", "service_state")]
train = to_numeric_clean(train, numeric_cols)
test  = to_numeric_clean(test, numeric_cols)

# Drop fully-duplicate rows (optional but common)
train = train.drop_duplicates()
test  = test.drop_duplicates()

# Fill NaNs after coercion
train[numeric_cols] = train[numeric_cols].fillna(0)
test[numeric_cols]  = test[numeric_cols].fillna(0)

# Enforce non-negativity on count/size/rate-like fields
nonneg_cols = [
    "flow_duration",
    "fwd_packet_count", "bwd_packet_count",
    "fwd_bytes", "bwd_bytes",
    "packet_rate", "byte_rate", "avg_packet_size",
    "syn_flag_count", "rst_flag_count", "ack_flag_count"
]
train = clip_nonnegative(train, nonneg_cols)
test  = clip_nonnegative(test, nonneg_cols)

# Cast count-like columns to ints safely
count_like = [
    "fwd_packet_count", "bwd_packet_count", "fwd_bytes", "bwd_bytes",
    "syn_flag_count", "rst_flag_count", "ack_flag_count"
]
for c in count_like:
    train[c] = np.rint(train[c]).astype("int64")
    test[c]  = np.rint(test[c]).astype("int64")

# =========================
# Outlier capping (TRAIN-only, avoids leakage)
# =========================
cap_cols = ["flow_duration", "packet_rate", "byte_rate", "avg_packet_size"]
train, test, bounds = cap_outliers_iqr(train, test, cap_cols, k=3.0)

# =========================
# Label cleanup (if present)
# =========================
if LABEL_COL in train.columns:
    train[LABEL_COL] = train[LABEL_COL].astype("string").str.strip()
if LABEL_COL in test.columns:
    test[LABEL_COL] = test[LABEL_COL].astype("string").str.strip()

# =========================
# Save
# =========================
train.to_csv(TRAIN_OUT, index=False)
test.to_csv(TEST_OUT, index=False)

print("Cleaning complete.")
print("Train:", train.shape, "Test:", test.shape)
print("Outlier bounds used (train-derived):", bounds)
print("Saved:", TRAIN_OUT, "and", TEST_OUT)

KeyboardInterrupt: 

In [3]:
import pandas as pd
dataframe1 = pd.read_csv("Cleaned Data/cic_ids2017_common_train_clean.csv")
dataframe1

,flow_duration,protocol,service_state,fwd_packet_count,bwd_packet_count,fwd_bytes,bwd_bytes,packet_rate,byte_rate,avg_packet_size,syn_flag_count,rst_flag_count,ack_flag_count,label
0,24609.0,udp_or_tcp,dns,2,2,62,378,162.542159,17879.637530,117.750000,0,0,0,BENIGN
1,244.0,tcp,ftp,2,1,14,0,453.442055,37014.426841,9.333333,1,0,1,FTP-Patator
2,54981037.0,tcp,http,8,6,334,11595,0.207399,176.718535,852.071429,0,0,1,DoS Hulk
3,225.0,udp_or_tcp,dns,2,2,66,264,453.442055,37014.426841,90.750000,0,0,0,BENIGN
4,4905.0,unknown,other,29,1,57604,6,453.442055,37014.426841,1007.266667,0,0,1,BENIGN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1492347,31195.0,udp_or_tcp,dns,4,2,152,188,192.338516,10899.182560,63.000000,0,0,0,BENIGN
1492348,24335.0,udp_or_tcp,dns,2,2,68,182,164.372303,10273.268954,71.000000,0,0,0,BENIGN
1492349,54981037.0,tcp,http,5,6,304,11595,0.110743,119.793997,1007.266667,0,0,0,DoS Hulk
1492350,242.0,tcp,http,4,0,0,0,453.442055,0.000000,0.000000,0,0,1,DoS Hulk


In [4]:
dataframe1['label'].value_counts()

label
BENIGN                        1229147
DoS Hulk                       133626
DDoS                           101884
DoS GoldenEye                    8227
DoS Slowhttptest                 4162
DoS slowloris                    3970
FTP-Patator                      3420
PortScan                         2566
SSH-Patator                      2525
Web Attack � Brute Force         1149
Bot                              1099
Web Attack � XSS                  522
Infiltration                       29
Web Attack � Sql Injection         16
Heartbleed                          9
Label                               1
Name: count, dtype: int64

In [ ]:
dataframe1.isna().sum()

In [6]:
dataframe2 = pd.read_csv("Cleaned Data/cic_IoT2023_common_train_clean.csv")
dataframe2

,flow_duration,protocol,service_state,fwd_packet_count,bwd_packet_count,fwd_bytes,bwd_bytes,packet_rate,byte_rate,avg_packet_size,syn_flag_count,rst_flag_count,ack_flag_count,label
0,0.000000,60,http,5,5,472,472,23.671858,0.000000,7.168421,0,0,0,DDoS-ACK_Fragmentation
1,0.000000,60,http,5,5,27,27,2.393046,0.000000,5.684211,1,0,0,DDoS-SYN_Flood
2,0.033982,611,http,5,5,27,27,1.192715,1597.595463,5.714737,0,0,0,DDoS-PSHACK_Flood
3,0.000000,470,http,5,5,296,296,9.841972,0.000000,7.168421,0,0,0,Mirai-greeth_flood
4,1.089783,60,http,5,5,27,27,0.506993,13.688810,5.684211,2,0,0,DoS-SYN_Flood
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4002583,0.349406,170,http,5,5,25,25,5013.985490,143.099810,5.263158,0,0,0,DDoS-UDP_Flood
4002584,0.804792,611,http,5,5,27,27,326.714370,67.657192,5.731579,2,0,0,DDoS-SynonymousIP_Flood
4002585,0.105498,170,http,5,5,25,25,5013.985490,473.944823,5.263158,0,0,0,DDoS-UDP_Flood
4002586,0.324876,60,http,5,5,27,27,3.696571,166.217258,5.684211,1,0,0,DDoS-SynonymousIP_Flood


In [6]:
dataframe2['label'].value_counts()

label
DDoS-UDP_Flood             622030
DDoS-SynonymousIP_Flood    376052
DDoS-ICMP_Flood            370775
DoS-UDP_Flood              364757
DDoS-SYN_Flood             339126
DDoS-TCP_Flood             316983
DDoS-PSHACK_Flood          286650
DoS-TCP_Flood              252741
DDoS-RSTFINFlood           232610
DoS-SYN_Flood              212411
BenignTraffic              126389
Mirai-udpplain             102301
Mirai-greeth_flood          93916
Mirai-greip_flood           75171
DDoS-ICMP_Fragmentation     50850
MITM-ArpSpoofing            35394
DDoS-UDP_Fragmentation      33323
DDoS-ACK_Fragmentation      31672
DNS_Spoofing                20700
Recon-HostDiscovery         15326
Recon-OSScan                11153
Recon-PortScan               9223
DoS-HTTP_Flood               8263
VulnerabilityScan            4289
DDoS-HTTP_Flood              3291
DDoS-SlowLoris               2691
DictionaryBruteForce         1488
BrowserHijacking              646
CommandInjection              609
SqlInjec

In [7]:
dataframe2.isna().sum()

flow_duration       0
protocol            0
service_state       0
fwd_packet_count    0
bwd_packet_count    0
fwd_bytes           0
bwd_bytes           0
packet_rate         0
byte_rate           0
avg_packet_size     0
syn_flag_count      0
rst_flag_count      0
ack_flag_count      0
label               0
dtype: int64

In [7]:
dataframe3 = pd.read_csv("Cleaned Data/NSL_KDD_common_test_clean.csv")
dataframe3

,flow_duration,protocol,service_state,fwd_packet_count,bwd_packet_count,fwd_bytes,bwd_bytes,packet_rate,byte_rate,avg_packet_size,syn_flag_count,rst_flag_count,ack_flag_count,class,label_bin
0,0.0,icmp,ecr_i_sf,0,0,1032,0,0.0,0.0,79.384615,0,0,0,anomaly,1
1,0.0,tcp,http_rej,0,0,0,0,0.0,0.0,0.000000,0,0,0,normal,0
2,0.0,tcp,private_s0,0,0,0,0,0.0,0.0,0.000000,0,0,0,anomaly,1
3,0.0,udp,domain_u_sf,0,0,45,114,0.0,0.0,79.500000,0,0,0,normal,0
4,0.0,tcp,http_s0,0,0,0,0,0.0,0.0,0.000000,0,0,0,anomaly,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2798,0.0,tcp,http_sf,0,0,217,23622,0.0,0.0,2979.875000,0,0,0,normal,0
2799,0.0,tcp,smtp_sf,0,0,1087,302,0.0,0.0,1389.000000,0,0,0,normal,0
2800,0.0,tcp,http_sf,0,0,394,5643,0.0,0.0,2012.333333,0,0,0,normal,0
2801,0.0,tcp,http_sf,0,0,147,11633,0.0,0.0,2945.000000,0,0,0,normal,0


In [9]:
dataframe3['class'].value_counts()

class
normal     2444
anomaly     359
Name: count, dtype: int64

In [8]:
dataframe3['label_bin'].value_counts()

label_bin
0    2444
1     359
Name: count, dtype: int64

In [12]:
dataframe3.isna().sum()

flow_duration      0
protocol           0
service_state      0
fwd_bytes          0
bwd_bytes          0
packet_rate        0
byte_rate          0
avg_packet_size    0
class              0
label_bin          0
dtype: int64

In [10]:
dataframe4 = pd.read_csv("Cleaned Data/UNSW_common_train_clean.csv")
dataframe4

,flow_duration,protocol,service_state,fwd_packet_count,bwd_packet_count,fwd_bytes,bwd_bytes,packet_rate,byte_rate,avg_packet_size,syn_flag_count,ack_flag_count,label,attack_cat
0,0.121478,tcp,none_fin,6,4,258,172,74.087490,3539.735590,43.000000,0,0,0,Normal
1,0.649902,tcp,none_fin,14,38,734,42014,78.473372,65776.070854,822.076923,0,0,0,Normal
2,1.623129,tcp,none_fin,8,16,364,13186,14.170161,8348.073382,564.583333,1,1,0,Normal
3,1.681642,tcp,ftp_fin,12,12,628,770,13.677108,831.330331,58.250000,0,0,0,Normal
4,0.449454,tcp,none_fin,10,6,534,268,33.373826,1784.387279,50.125000,1,1,0,Normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
86793,1.914309,tcp,smtp_fin,74,30,69997,2132,53.805315,37678.870026,693.548077,1,1,1,Exploits
86794,3.719110,tcp,none_fin,66,340,3086,426483,108.897021,115503.171458,1058.051724,1,1,1,Exploits
86795,0.996503,tcp,pop3_fin,20,30,942,16684,49.171955,17687.854427,352.520000,1,1,1,Exploits
86796,1.557125,tcp,smtp_fin,28,22,12601,1954,31.468251,9347.354901,291.100000,1,1,1,Exploits


In [11]:
dataframe4['attack_cat'].value_counts()

attack_cat
Normal            41404
Exploits          19184
Fuzzers           13346
Reconnaissance     5820
DoS                3218
Backdoor            987
Analysis            965
Shellcode           954
Generic             797
Worms               123
Name: count, dtype: int64

In [13]:
dataframe4.isna().sum()

flow_duration       0
protocol            0
service_state       0
fwd_packet_count    0
bwd_packet_count    0
fwd_bytes           0
bwd_bytes           0
packet_rate         0
byte_rate           0
avg_packet_size     0
syn_flag_count      0
ack_flag_count      0
label               0
attack_cat          0
dtype: int64

In [19]:
dataframe4.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 86798 entries, 0 to 86797
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   flow_duration     86798 non-null  float64
 1   protocol          86798 non-null  object 
 2   service_state     86798 non-null  object 
 3   fwd_packet_count  86798 non-null  int64  
 4   bwd_packet_count  86798 non-null  int64  
 5   fwd_bytes         86798 non-null  int64  
 6   bwd_bytes         86798 non-null  int64  
 7   packet_rate       86798 non-null  float64
 8   byte_rate         86798 non-null  float64
 9   avg_packet_size   86798 non-null  float64
 10  syn_flag_count    86798 non-null  int64  
 11  ack_flag_count    86798 non-null  int64  
 12  label             86798 non-null  int64  
 13  attack_cat        86798 non-null  object 
dtypes: float64(4), int64(7), object(3)
memory usage: 9.3+ MB


In [23]:
import pandas as pd
dataframe1 = pd.read_csv("Datasets/CIC_IoT2023_train.csv")
dataframe1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5491971 entries, 0 to 5491970
Data columns (total 47 columns):
 #   Column           Dtype  
---  ------           -----  
 0   flow_duration    float64
 1   Header_Length    float64
 2   Protocol Type    float64
 3   Duration         float64
 4   Rate             float64
 5   Srate            float64
 6   Drate            float64
 7   fin_flag_number  float64
 8   syn_flag_number  float64
 9   rst_flag_number  float64
 10  psh_flag_number  float64
 11  ack_flag_number  float64
 12  ece_flag_number  float64
 13  cwr_flag_number  float64
 14  ack_count        float64
 15  syn_count        float64
 16  fin_count        float64
 17  urg_count        float64
 18  rst_count        float64
 19  HTTP             float64
 20  HTTPS            float64
 21  DNS              float64
 22  Telnet           float64
 23  SMTP             float64
 24  SSH              float64
 25  IRC              float64
 26  TCP              float64
 27  UDP         

In [24]:
dataframe1.isna().sum()

flow_duration      0
Header_Length      0
Protocol Type      0
Duration           0
Rate               0
Srate              0
Drate              0
fin_flag_number    0
syn_flag_number    0
rst_flag_number    0
psh_flag_number    0
ack_flag_number    0
ece_flag_number    0
cwr_flag_number    0
ack_count          0
syn_count          0
fin_count          0
urg_count          0
rst_count          0
HTTP               0
HTTPS              0
DNS                0
Telnet             0
SMTP               0
SSH                0
IRC                0
TCP                0
UDP                0
DHCP               0
ARP                0
ICMP               0
IPv                0
LLC                0
Tot sum            0
Min                0
Max                0
AVG                0
Std                0
Tot size           0
IAT                0
Number             0
Magnitue           0
Radius             0
Covariance         0
Variance           0
Weight             0
label              0
dtype: int64